In [52]:
# Python 3.10.11
# %pip install -r requirements.txt > /dev/null
from args import *
from utils import *

In [53]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

# TODO(241225) 依赖导入
import pandas as pd


In [54]:
# TODO(241225) 导入label数据
label_df = pd.read_csv(sample_labels_file_path, index_col=0)

# 裁减样本数量，使得 0 - 1 样本数量一致
df_0 = label_df[label_df['label'] == 0]
df_1 = label_df[label_df['label'] == 1]
min_len = min(len(df_0), len(df_1))
new_df = pd.concat([df_0[:min_len], df_1[:min_len]], axis=0)

sample_key_list = label_df.index.to_list()
sample_key_list = new_df.index.to_list()

print(f"label 样本数量: {len(sample_key_list)}")

# TODO(241225) 导入gene数据
# 根据 sample_key_list 为基准, 若模态数据中不存在 sample_key 则填充新数据
gene_array = load_gene_data_by_sample_key(sample_key_list).values
cnv_array = load_cnv_data_by_sample_key(sample_key_list).values

wsi_array = load_wsi_data_by_sample_key(sample_key_list)
report_array = load_report_data_by_sample_key(sample_key_list)

label_array = label_df.values
label_array = new_df.values

_, gene_dim = gene_array.shape
_, cnv_dim = cnv_array.shape
_, wsi_dim = wsi_array.shape
_, report_dim = report_array.shape
_, label_dim = label_array.shape

print(f"""
gene 数据维度:   {gene_dim}
cnv 数据维度:    {cnv_dim}
wsi 数据维度:    {wsi_dim}
report 数据维度: {report_dim}
""")

batch_size = 32

all_dataset = MultiOmicsDataset(gene_array, cnv_array, report_array, wsi_array, label_array)

# # 假设 all_dataset 是一个 Dataset 对象
# train_len = int(len(all_dataset) * 0.8)  # 80% 的数据用作训练集
# test_len = len(all_dataset) - train_len  # 剩余的数据用作验证集

# # 使用 random_split 分割数据集
# train_val_dataset, test_dataset = random_split(all_dataset, [train_len, test_len])

train_loader = DataLoader(all_dataset, batch_size=batch_size, shuffle=True, num_workers=3, drop_last=False)
val_loader = DataLoader(all_dataset, batch_size=batch_size, shuffle=True, num_workers=3, drop_last=False)

label 样本数量: 152

gene 数据维度:   2340
cnv 数据维度:    2340
wsi 数据维度:    2048
report 数据维度: 768



In [153]:
report_array.shape

(152, 768)

In [241]:


def report_loader_layer():
    model_file_path = pkg_dir_path.parent / "report/data/output/bert.pth"
    assert model_file_path.exists()

    model = torch.load(model_file_path).to(device)
    model = model.classifier

    # # 冻结参数
    # # 冻结第一层
    # for param in model[0].parameters():
    #     param.requires_grad = False

    # # 确认第一层的参数不需要梯度
    # for name, param in model.named_parameters():
    #     if name.startswith('0.'):
    #         print(name, param.requires_grad)  # 应该输出False

    return model

report_loader_layer = report_loader_layer()

/tmp/ipykernel_1106/822536543.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_file_path).to(device)


In [242]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import models  # 如果需要使用预训练模型

# 假设我们使用一个预训练的ResNet作为特征提取器，并添加自定义层
class CustomModel(nn.Module):
    def __init__(self, base_model, num_classes):
        super(CustomModel, self).__init__()
        # 假设base_model期望的输入形状是[batch_size, 3, 224, 224]
        # 我们将第一层替换为一个全连接层
        self.base_model = base_model
        # self.custom_layer = nn.Sequential(
        #     nn.Linear(base_model.in_features, 256),
        #     nn.ReLU(),
        #     # nn.Dropout(0.5),
        #     nn.Linear(256, num_classes)
        # )

    def forward(self, x):
        x = self.base_model(x)
        # x = self.custom_layer(x)
        return x

# 加载预训练模型并替换最后一层
num_classes = 1  # 根据您的任务确定类别数
model = CustomModel(report_loader_layer, num_classes).to(device)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 适用于二分类问题
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 学习率调度器，可以在训练过程中调整学习率
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min')

In [243]:
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

from torchmetrics import AUROC
from torch.utils.tensorboard import SummaryWriter

# 假设以下变量已经在代码的其他部分被正确定义和初始化
# num_epochs, test_loader, device, model, writer

# 初始化 AUC 计算器，指定任务类型为二分类
# auc = AUROC(task='binary')

In [244]:


        with torch.no_grad():
            if batch:
            # for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                outputs = model(input_ids, attention_mask=attention_mask)
                _, preds = torch.max(outputs.logits, dim=1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()
    

                # 保存预测值和标签
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())


        # 计算准确率
        accuracy = correct / total
        accuracies.append(accuracy)

        
        epoch_accuracy = sum(accuracies) / len(accuracies)
        print(f"Validation Accuracy: {epoch_accuracy} ({accuracy})")
        # 记录验证过程中的准确率
        writer.add_scalar('validation_accuracy', epoch_accuracy, epoch)


        # 计算 AUC
        auc_value = auc(torch.tensor(all_preds), torch.tensor(all_labels)).item()  # 使用.item()获取标量值
        auces.append(auc_value)
        epoch_auc = sum(auces) / len(auces)
        print(f"Validation AUC: {epoch_auc}")
        writer.add_scalar('validation_auc', epoch_auc, epoch)
    

writer.close()


KeyError: 'input_ids'

In [247]:
num_epochs = 5
for epoch in range(num_epochs):


    model.train()
    # 导入 batch 数据
    for batch in train_loader:
        gene_tensor = batch["gene_tensor"].to(device)
        cnv_tensor = batch["cnv_tensor"].to(device)
        report_tensor = batch["report_tensor"].to(device)
        wsi_tensor = batch["wsi_tensor"].to(device)
        label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)
        label_tensor = label_tensor.squeeze()
        label_tensor = label_tensor.long()
        # outputs = model(gene_tensor, cnv_tensor, report_tensor, wsi_tensor)
        outputs = model(report_tensor)  # 假设wsi_tensor是主要的输入特征
        # print((outputs.shape, label_tensor.shape))

        optimizer.zero_grad()
        loss = criterion(outputs, label_tensor)
        loss.backward()
        optimizer.step()

    print("train loss: ", loss.item())
    # 更新学习率
    # scheduler.step(loss)





    correct = 0
    total = 0
    # 初始化变量来存储所有预测值和标签
    all_preds = []
    all_labels = []
    

    # 在每个epoch结束时进行验证
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            # ...（数据加载代码保持不变）...

            outputs = model(report_tensor)

            _pred, preds = torch.max(outputs, 1)  # 获取预测的类别
            # all_preds.extend(_pred.cpu().numpy())  # 将预测值添加到列表中
            # all_labels.extend(label_tensor.cpu().numpy())  # 将真实标签添加到列表中

            all_preds = []
            all_labels = []
            from sklearn.metrics import roc_auc_score

            # 假设outputs是模型的输出，形状为[24, 2]
            # 我们只关心正类的概率，所以取第二列（索引为1）
            all_preds_prob = torch.sigmoid(outputs)[:, 1].cpu().numpy()
            all_labels.extend(label_tensor.cpu().numpy())

            # 计算AUC
            auc = roc_auc_score(all_labels, all_preds_prob)

            print(f'AUC: {auc}')


            total += label_tensor.size(0)
            correct += (preds == label_tensor).sum().item()


    # 计算准确率
    accuracy = correct / total
    print(f"Validation Accuracy: ({accuracy})")


    # # 计算 AUC
    # auc_value = auc(torch.tensor(all_preds), torch.tensor(all_labels))  # 使用.item()获取标量值
    # print(f"Validation AUC: {auc_value}")

 

train loss:  0.6420381665229797
AUC: 0.8333333333333335
AUC: 0.8333333333333335
AUC: 0.8333333333333335
AUC: 0.8333333333333335
AUC: 0.8333333333333335
Validation Accuracy: (0.625)
train loss:  0.7107751965522766
AUC: 0.5244755244755245
AUC: 0.5244755244755245
AUC: 0.5244755244755245
AUC: 0.5244755244755245
AUC: 0.5244755244755245
Validation Accuracy: (0.5416666666666666)
train loss:  0.7105150818824768
AUC: 0.5314685314685315
AUC: 0.5314685314685315
AUC: 0.5314685314685315
AUC: 0.5314685314685315
AUC: 0.5314685314685315
Validation Accuracy: (0.5416666666666666)
train loss:  0.6093707084655762
AUC: 0.84375
AUC: 0.84375
AUC: 0.84375
AUC: 0.84375
AUC: 0.84375
Validation Accuracy: (0.7083333333333334)
train loss:  0.6355727314949036
AUC: 0.7999999999999999
AUC: 0.7999999999999999
AUC: 0.7999999999999999
AUC: 0.7999999999999999
AUC: 0.7999999999999999
Validation Accuracy: (0.6666666666666666)


In [238]:
torch.sigmoid(outputs)[:, 1]


tensor([0.9373, 0.9253, 0.3072, 0.0723, 0.2690, 0.7682, 0.5695, 0.6142, 0.6570,
        0.4753, 0.5359, 0.8659, 0.9696, 0.6331, 0.5043, 0.5691, 0.8155, 0.1218,
        0.5368, 0.7958, 0.5486, 0.2963, 0.9348, 0.1828], device='cuda:0')

In [240]:
outputs

tensor([[-3.0180,  2.7038],
        [-2.8495,  2.5167],
        [ 0.8201, -0.8131],
        [ 2.7710, -2.5514],
        [ 1.1250, -0.9995],
        [-1.4267,  1.1981],
        [-0.2943,  0.2798],
        [-0.5096,  0.4648],
        [-0.7202,  0.6500],
        [ 0.1254, -0.0988],
        [-0.1758,  0.1437],
        [-2.2301,  1.8648],
        [-3.8425,  3.4613],
        [-0.5987,  0.5453],
        [-0.1124,  0.0171],
        [-0.2896,  0.2782],
        [-1.6918,  1.4859],
        [ 2.1636, -1.9752],
        [-0.1528,  0.1476],
        [-1.5461,  1.3601],
        [-0.2133,  0.1950],
        [ 0.9125, -0.8650],
        [-3.0078,  2.6635],
        [ 1.6566, -1.4974]], device='cuda:0')

In [239]:
torch.sigmoid(outputs)

tensor([[-3.0180,  2.7038],
        [-2.8495,  2.5167],
        [ 0.8201, -0.8131],
        [ 2.7710, -2.5514],
        [ 1.1250, -0.9995],
        [-1.4267,  1.1981],
        [-0.2943,  0.2798],
        [-0.5096,  0.4648],
        [-0.7202,  0.6500],
        [ 0.1254, -0.0988],
        [-0.1758,  0.1437],
        [-2.2301,  1.8648],
        [-3.8425,  3.4613],
        [-0.5987,  0.5453],
        [-0.1124,  0.0171],
        [-0.2896,  0.2782],
        [-1.6918,  1.4859],
        [ 2.1636, -1.9752],
        [-0.1528,  0.1476],
        [-1.5461,  1.3601],
        [-0.2133,  0.1950],
        [ 0.9125, -0.8650],
        [-3.0078,  2.6635],
        [ 1.6566, -1.4974]], device='cuda:0')

In [225]:
auc = roc_auc_score(all_labels, all_preds_prob)

TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.